# Pipes And Redirection

## Quick notes

### What this covers
- The three standard streams (stdin 0, stdout 1, stderr 2) and file descriptors in general.
- Redirection operators: `>`, `>>`, `<`, `2>`, `2>&1`, `&>`, `>|`, `<<`, `<<<`, `<( )`.
- Why the order of redirections matters, and why a pipe has its own exit-status rules.
- Reading input safely, the "pipeline runs in a subshell" trap, named pipes, `xargs`, `tee`.

### Key points
- Every process starts with fd 0 (input), fd 1 (normal output) and fd 2 (errors). Redirection just re-points these numbers.
- `cmd > file` replaces the file, `cmd >> file` appends, `cmd < file` feeds the file in.
- Redirections are processed left to right. `>f 2>&1` sends both to `f`; `2>&1 >f` does not.
- A pipeline's exit status is the status of the LAST command (unless `set -o pipefail`). `PIPESTATUS` keeps all of them.
- Here-doc (`<<`) and here-string (`<<<`) feed text to stdin without a temp file.
- Process substitution `<( cmd )` makes a command's output look like a file name.

### Gotchas (the part worth revising twice)
- `sort file > file` empties the file BEFORE sort reads it. Write to a temp file, then `mv`.
- `cmd | while read ...` runs the loop in a subshell: variables set inside vanish. Use `< <(cmd)` or a file.
- `2>&1 >file` is NOT the same as `>file 2>&1`.
- bash 3.2 (macOS default) has `&>` but NOT `&>>`, `|&`, `coproc` or `mapfile`. Write `>> f 2>&1` instead.
- `read` without `-r` eats backslashes; without `IFS=` it strips leading/trailing blanks.
- BSD `xargs` (macOS) does NOT run the command when the input is empty; GNU `xargs` runs it once unless you add `-r`. Same script, different behaviour.
- A FIFO blocks until both a reader and a writer exist. Always put one side in the background.

### When it matters
- Logging scripts (stdout vs stderr), cron jobs, CI output, safe in-place file edits, and any pipeline that must report failures.

### One-line recap example
```bash
some_cmd >out.log 2>&1   # both streams to one file, in that order
```

## 01. The three standard streams

- stdin is fd 0, stdout is fd 1, stderr is fd 2.
- Programs write normal results to fd 1 and complaints to fd 2, so you can split them.
- `>&2` means "send this to wherever fd 2 goes".
- In a notebook both show up in the output, in order; in a real terminal they look the same too, but they are separate streams.

In [1]:
# setup: a %%bash cell magic that keeps ONE bash session alive between cells
import atexit, os, shutil, subprocess, tempfile, uuid
from IPython.core.magic import register_cell_magic

_WORK = tempfile.mkdtemp(prefix="bashnb_")
atexit.register(shutil.rmtree, _WORK, ignore_errors=True)
_ENV = {k: v for k, v in os.environ.items() if k not in ("CLICOLOR", "CLICOLOR_FORCE", "FORCE_COLOR", "PYDEVD_DISABLE_FILE_VALIDATION")}
_ENV.update(LC_ALL="C", TZ="UTC", HOME=_WORK, PS1="", PS4="+ ")
_session = None

def _start():
    global _session
    _session = subprocess.Popen(["bash", "--noprofile", "--norc"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=_WORK, env=_ENV)

def _clean(text):
    return text.replace(os.path.join(_WORK, "script"), "script").replace(_WORK, "<work>")

def _run_persistent(code, timeout=60):
    global _session
    if _session is None or _session.poll() is not None:
        _start()
    token = uuid.uuid4().hex
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    _session.stdin.write(". '%s' </dev/null; printf '\\n%s %%d\\n' $?\n" % (path, token))
    _session.stdin.flush()
    out, status = [], None
    import threading
    result = {}
    def reader():
        while True:
            line = _session.stdout.readline()
            if not line:
                result["status"] = None; return
            if line.startswith(token + " "):
                result["status"] = int(line.split()[1]); return
            out.append(line)
    t = threading.Thread(target=reader, daemon=True); t.start(); t.join(timeout)
    os.remove(path)
    if t.is_alive():
        _session.kill(); _session = None
        return "".join(out) + "[timed out]\n", 124
    if result.get("status") is None:       # the shell itself exited (e.g. the cell ran `exit N`)
        code = _session.wait(); _session = None
        return "".join(out) + "(the bash session ended; the next cell starts a fresh one)\n", code
    text = "".join(out)
    if text.endswith("\n\n"):
        text = text[:-1]          # drop the blank line our status marker adds
    return text, result.get("status") if result.get("status") is not None else 1

def _run_fresh(code, timeout=60):
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    done = subprocess.run(["bash", "--noprofile", "--norc", path], stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, cwd=_WORK, env=_ENV, timeout=timeout)
    os.remove(path)
    return done.stdout, done.returncode

@register_cell_magic
def bash(line, cell):
    """Run the cell in bash. Flags: fresh (new shell), expect_fail (show non-zero status instead of raising)."""
    flags = line.split()
    text, status = (_run_fresh if "fresh" in flags else _run_persistent)(cell)
    print(_clean(text), end="" if text.endswith("\n") or not text else "\n")
    if status != 0:
        if "expect_fail" in flags:
            print("[exit status %d]" % status)
        else:
            raise RuntimeError("bash cell exited with status %d" % status)

In [2]:
%%bash
mkdir s01 && cd s01
echo "normal result"             # fd 1
echo "a complaint" >&2           # fd 2
# Capture only stdout; stderr still reaches the screen:
{ echo out; echo err >&2; } > only_out.txt
echo "--- only_out.txt contains:"
cat only_out.txt
cd ..

normal result
a complaint
err
--- only_out.txt contains:
out


## 02. `>` `>>` `<` and `2>`

- `>` creates or TRUNCATES the file. `>>` appends. `<` reads from a file.
- `2>` redirects only stderr. `1>` is the same as `>`.
- The redirection is done by the shell before the command starts, so the command never sees it.

In [3]:
%%bash
cd s01
echo "first"  >  log.txt        # overwrite / create
echo "second" >> log.txt        # append
echo "third"  >> log.txt
cat log.txt
echo "--- count lines read from a file with <:"
wc -l < log.txt                # wc sees no file name, so prints only the number
echo "--- with a file argument wc also prints the name:"
wc -l log.txt
echo "--- stderr to a file:"
ls /definitely/not/here 2> err.txt
echo "exit=$?"
cat err.txt
cd ..

first
second
third
--- count lines read from a file with <:
       3
--- with a file argument wc also prints the name:
       3 log.txt
--- stderr to a file:
exit=1
ls: /definitely/not/here: No such file or directory


## 03. Order matters: `>f 2>&1` vs `2>&1 >f`

- `2>&1` means "make fd 2 a copy of whatever fd 1 points to RIGHT NOW".
- `>f 2>&1`: first stdout goes to f, then stderr copies it, so BOTH go to f.
- `2>&1 >f`: stderr copies the old stdout (the screen) first, then stdout moves to f. Errors stay on screen.
- Remember: read redirections left to right.

In [4]:
%%bash
cd s01
good()  { ls /no/such/thing >both.txt 2>&1; }          # both in the file
bad()   { ls /no/such/thing 2>&1 >only_stdout.txt; }   # error escapes to the screen
good
echo "good: file has -> $(cat both.txt)"
echo "bad: (the next line is the error leaking to the screen)"
bad
echo "bad: file has -> [$(cat only_stdout.txt)]"
cd ..

good: file has -> ls: /no/such/thing: No such file or directory
bad: (the next line is the error leaking to the screen)
ls: /no/such/thing: No such file or directory
bad: file has -> []


## 04. `&>`, `/dev/null` and what bash 3.2 lacks

- `&>file` sends both stdout and stderr to the file (bash extension; works in 3.2).
- `&>>file` (append both) needs bash 4+. On 3.2 write `>>file 2>&1`.
- `/dev/null` is a device that swallows everything and reads as empty. Use it to silence output.
- A common pattern: `cmd >/dev/null 2>&1` (portable to plain `sh`) or `cmd &>/dev/null` (bash only).
- `&>>` is a syntax error on 3.2, so it is shown below in a throwaway shell.

In [5]:
%%bash
cd s01
ls /no/such/thing &> amp.txt
echo "&> captured: $(cat amp.txt)"
ls /no/such/thing >> amp.txt 2>&1          # portable append of both
echo "lines now: $(wc -l < amp.txt | tr -d ' ')"
ls /no/such/thing >/dev/null 2>&1
echo "silenced, status still visible: $?"
echo "reading from /dev/null gives EOF at once: [$(cat /dev/null)]"
cd ..

&> captured: ls: /no/such/thing: No such file or directory
lines now: 2
silenced, status still visible: 1
reading from /dev/null gives EOF at once: []


In [6]:
%%bash fresh expect_fail
# needs bash 4+ for &>> -- on 3.2 the whole line fails to parse
bash -c 'echo hi &>> /dev/null' 2>&1 | sed 's/^[^:]*: //'
echo "bash version major: ${BASH_VERSINFO[0]}"

-c: line 0: syntax error near unexpected token `>'
-c: line 0: `echo hi &>> /dev/null'
bash version major: 3


## 05. `tee`: split a stream

- `tee file` copies stdin to the file AND to stdout, like a T-junction in a pipe.
- `tee -a` appends instead of truncating (both BSD and GNU).
- `tee` only splits stdout. To also catch errors, put `2>&1` before the pipe.
- Classic use: save a log while still watching it, or `sudo tee` to write a root-owned file.

In [7]:
%%bash
cd s01
printf 'alpha\nbeta\n' | tee copy1.txt copy2.txt     # two files plus the screen
echo "--- copy2.txt:"; cat copy2.txt
echo gamma | tee -a copy1.txt > /dev/null             # append, hide the screen copy
echo "--- copy1.txt after -a:"; cat copy1.txt
echo "--- errors need 2>&1 before the pipe:"
{ echo ok; echo oops >&2; } 2>&1 | tee both_tee.txt > /dev/null
cat both_tee.txt
cd ..

alpha
beta
--- copy2.txt:
alpha
beta
--- copy1.txt after -a:
alpha
beta
gamma
--- errors need 2>&1 before the pipe:
ok
oops


## 06. Pipes and exit status: `PIPESTATUS`

- `a | b | c` starts all three at once, connected by pipes.
- `$?` after a pipeline is the exit status of the LAST command only.
- So `false | true` looks like success. Hidden failures are the classic pipe bug.
- `${PIPESTATUS[@]}` holds the status of each command. It is overwritten by the very next command, so copy it right away.

In [8]:
%%bash
false | true
echo "last-command status: $?"
false | true
echo "PIPESTATUS: ${PIPESTATUS[*]}"
true | false | true
st=("${PIPESTATUS[@]}")           # copy immediately
echo "copied: ${st[*]} (count ${#st[@]})"
echo "--- grep finds nothing (status 1) but wc succeeds:"
printf 'a\nb\n' | grep zzz | wc -l
echo "PIPESTATUS: ${PIPESTATUS[*]}"

last-command status: 0
PIPESTATUS: 1 0
copied: 0 1 0 (count 3)
--- grep finds nothing (status 1) but wc succeeds:
       0
PIPESTATUS: 0 1 0


## 07. `set -o pipefail` (use a fresh shell)

- With `pipefail`, a pipeline fails if ANY command fails; the status is that of the right-most failing command.
- Combine with `set -e` in scripts to stop on a broken pipe stage.
- `pipefail` is a shell option: it would stay on for the rest of a notebook session, so we show it in a fresh shell.
- Beware: `grep` returns 1 for "no match" and that counts as failure under pipefail.

In [9]:
%%bash fresh
false | true;  echo "default:  $?"
set -o pipefail
false | true;  echo "pipefail: $?"
true | false | true; echo "middle fails: $?"
echo "--- grep with no match inside a pipeline:"
printf 'a\n' | grep zzz | wc -l
echo "status under pipefail: ${PIPESTATUS[*]}"

default:  0
pipefail: 1
middle fails: 1
--- grep with no match inside a pipeline:
       0
status under pipefail: 0 1 0


## 08. Here-documents `<<`

- `cmd <<WORD` feeds the following lines (up to a line that is exactly WORD) to stdin.
- Unquoted delimiter: the body is expanded (`$var`, `$(cmd)`; a backslash before `$`, a backtick or `\` makes it literal). Quoted (`<<'WORD'` or `<<"WORD"`): taken literally.
- `<<-WORD` strips ALL leading TABS (not spaces) from the body and the delimiter line, handy for indented code. Tab-based indentation inside the body is lost too.
- The closing delimiter must be alone on its line (no trailing spaces).

In [10]:
%%bash
name="world"
echo "--- unquoted: variables and commands expand"
cat <<EOF
hello $name
today is fixed: $(echo 2024-01-01)
a literal dollar: \$HOME
EOF
echo "--- quoted: nothing expands"
cat <<'EOF'
hello $name
$(echo not run)
EOF

--- unquoted: variables and commands expand
hello world
today is fixed: 2024-01-01
a literal dollar: $HOME
--- quoted: nothing expands
hello $name
$(echo not run)


In [11]:
%%bash
# <<- removes leading TABS only (all of them, so the nested line below loses its extra tab). The body is indented with real tabs.
if true; then
	cat <<-EOF
	indented with tabs
		still nested one level
	EOF
fi
echo "--- same text, spaces are NOT stripped:"
cat <<-EOF
    four spaces stay
EOF
echo "--- feeding a here-doc to a command, then saving it:"
sort <<EOF > sorted_heredoc.txt
pear
apple
fig
EOF
cat sorted_heredoc.txt

indented with tabs
still nested one level
--- same text, spaces are NOT stripped:
    four spaces stay
--- feeding a here-doc to a command, then saving it:
apple
fig
pear


## 09. Here-strings `<<<`

- `cmd <<< "text"` gives the text (plus a trailing newline) to stdin.
- Shorter than `echo "text" | cmd`, and it does NOT create a pipeline subshell, so `read` can set your variables.
- Quote the string so spaces are kept.
- `<<<` is a bash feature, not POSIX `sh`.

In [12]:
%%bash
read -r first rest <<< "alpha beta gamma"
echo "first=$first rest=$rest"
echo "--- pipe version loses the variables:"
unset x y; echo "1 2" | read x y; echo "after pipe: x=[${x:-}] y=[${y:-}]"
echo "--- here-string keeps them:"
read x y <<< "1 2"; echo "after <<<: x=[$x] y=[$y]"
tr 'a-z' 'A-Z' <<< "shout"
wc -c <<< "abc"        # 4: the three letters plus the newline here-string adds

first=alpha rest=beta gamma
--- pipe version loses the variables:
after pipe: x=[] y=[]
--- here-string keeps them:
after <<<: x=[1] y=[2]
SHOUT
       4


## 10. Process substitution `<( )`

- `<(cmd)` runs `cmd` and gives you a file name (like `/dev/fd/63`) you can read.
- Perfect for tools that want files: `diff <(a) <(b)`, `comm <(sort x) <(sort y)`.
- `>(cmd)` is the output twin: write to a file name, the data goes to `cmd`.
- The commands run in the background; the name is not a regular file (you cannot `seek` in it).
- Not available in POSIX `sh`.

In [13]:
%%bash
printf 'apple\nbanana\ncherry\n' > a.txt
printf 'banana\ncherry\ndate\n'  > b.txt
echo "--- diff of two sorted streams, no temp files:"
diff <(sort a.txt) <(sort b.txt)
echo "(diff exit status: $?)"
echo "--- comm needs sorted input; -12 keeps lines common to both:"
comm -12 <(sort a.txt) <(sort b.txt)
echo "--- only in a (-23):"
comm -23 <(sort a.txt) <(sort b.txt)
echo "--- what the name looks like (digits masked):"
echo <(true) | sed 's/[0-9][0-9]*$/N/'
echo "--- output side: write once, get a count and a copy"
printf 'x\ny\nz\n' | tee >(wc -l > count.txt) > copy_all.txt
sleep 1; echo "count: $(tr -d ' ' < count.txt)"

--- diff of two sorted streams, no temp files:
1d0
< apple
3a3
> date
(diff exit status: 1)
--- comm needs sorted input; -12 keeps lines common to both:
banana
cherry
--- only in a (-23):
apple
--- what the name looks like (digits masked):
/dev/fd/N
--- output side: write once, get a count and a copy
count: 3


## 11. `exec` redirections and custom file descriptors

- `exec >file` (no command) redirects the shell ITSELF from then on. Dangerous in an interactive or shared session, so we use fresh shells.
- You can open extra descriptors: `exec 3>file` (write), `exec 4<file` (read), `exec 3>&-` (close).
- `>&3` means "send this to fd 3". Handy for logs separate from stdout.
- Save and restore: `exec 5>&1` saves stdout, `exec 1>&5 5>&-` restores it.

In [14]:
%%bash fresh
cd "$HOME"
exec 3> report.txt              # open fd 3 for writing
echo "line A" >&3
echo "line B" >&3
exec 3>&-                       # close it
cat report.txt
echo "--- reading with fd 4:"
exec 4< report.txt
read -r l1 <&4
read -r l2 <&4
exec 4<&-
echo "read: $l1 / $l2"
echo "--- redirect the whole script, then restore:"
exec 5>&1                       # save stdout
exec > captured.txt             # from now on stdout goes to the file
echo "this goes to the file"
exec 1>&5 5>&-                  # restore and close the saved copy
echo "back on screen; file holds: $(cat captured.txt)"

line A
line B
--- reading with fd 4:
read: line A / line B
--- redirect the whole script, then restore:
back on screen; file holds: this goes to the file


## 12. Reading lines safely: `IFS= read -r`

- Plain `read line` trims leading/trailing whitespace and treats `\` as an escape. That quietly corrupts data.
- Safe form: `while IFS= read -r line; do ... done < file`.
- `-r` keeps backslashes; `IFS=` (set only for that `read`) keeps spaces.
- The last line of a file with NO trailing newline makes `read` return failure while still filling the variable: add `|| [ -n "$line" ]`.

In [15]:
%%bash
printf '  indented\\tand a backslash\nnormal\n' > tricky.txt
echo "--- naive:"
while read line; do echo "[$line]"; done < tricky.txt
echo "--- safe:"
while IFS= read -r line; do echo "[$line]"; done < tricky.txt
echo "--- file without a final newline:"
printf 'one\ntwo-no-newline' > nonl.txt
while IFS= read -r line; do echo "plain loop got: $line"; done < nonl.txt
echo "--- with the || [ -n ] guard:"
while IFS= read -r line || [ -n "$line" ]; do echo "guarded loop got: $line"; done < nonl.txt

--- naive:
[indentedtand a backslash]
[normal]
--- safe:
[  indented\tand a backslash]
[normal]
--- file without a final newline:
plain loop got: one
--- with the || [ -n ] guard:
guarded loop got: one
guarded loop got: two-no-newline


## 13. `while read` from a pipe vs from a file (the subshell trap)

- Each side of a pipe runs in its own subshell. Variables changed inside die when the loop ends.
- Feeding the loop from a file (`< file`) or process substitution (`< <(cmd)`) or a here-string (`<<< "$(cmd)"`) keeps the loop in the current shell.
- bash 4.2+ has `shopt -s lastpipe` for this; on 3.2 it is an invalid option name. Prefer `< <(cmd)`.

In [16]:
%%bash
printf 'a\nb\nc\n' > three.txt
count=0
cat three.txt | while read -r l; do count=$((count+1)); done
echo "pipe:        count=$count   <- still 0, the loop was a subshell"
count=0
while read -r l; do count=$((count+1)); done < three.txt
echo "file:        count=$count"
count=0
while read -r l; do count=$((count+1)); done < <(printf 'x\ny\n')
echo "process sub: count=$count"
count=0
while read -r l; do count=$((count+1)); done <<< "$(printf 'p\nq\nr\ns\n')"
echo "here-string: count=$count"

pipe:        count=0   <- still 0, the loop was a subshell
file:        count=3
process sub: count=2
here-string: count=4


## 14. Named pipes (`mkfifo`)

- A FIFO is a pipe with a file name. One process writes, another reads, no data stored on disk.
- Opening a FIFO blocks until the other end opens too. If both ends are in the same foreground process it hangs forever.
- So start the writer in the background (`&`), then read.
- Always `rm` the FIFO afterwards. Here the demo is short and bounded.

In [17]:
%%bash
cd s01
mkfifo demo.fifo
( printf 'msg1\nmsg2\n' > demo.fifo ) &     # writer in the background
while IFS= read -r line; do echo "reader got: $line"; done < demo.fifo
wait
ls -l demo.fifo | cut -c1          # first character p = named pipe
rm demo.fifo
cd ..

reader got: msg1
reader got: msg2
p


## 15. `xargs`: turn lines into arguments

- Many commands take arguments, not stdin. `xargs` reads stdin and builds the command line.
- Default: input is split on blanks and newlines, and quotes and backslashes in it are treated specially, so names with spaces break. Use `-0` with NUL-separated input (`find -print0`, `printf '%s\0'`).
- `-n N` runs the command with N args at a time; `-I{}` puts each line where `{}` is (one run per line).
- Empty input: BSD `xargs` runs nothing; GNU `xargs` runs the command once unless `-r` is given (long form `--no-run-if-empty`, GNU only). BSD `xargs` accepts `-r` and ignores it, so `-r` is safe on both. Shown below.

In [18]:
%%bash
mkdir -p s15 && cd s15
touch "plain.txt" "with space.txt"
echo "--- splitting on blanks breaks names with spaces:"
printf 'with space.txt\n' | xargs -n1 echo "arg:"
echo "--- -0 keeps each name whole:"
printf 'with space.txt\0plain.txt\0' | xargs -0 -n1 echo "arg:"
echo "--- find | xargs -0 is the safe pairing:"
find . -type f -name '*.txt' -print0 | xargs -0 ls | sort
echo "--- -n 2 groups two per call:"
printf '1\n2\n3\n4\n5\n' | xargs -n 2 echo "batch:"
echo "--- -I{} one run per input line:"
printf 'a\nb\n' | xargs -I{} echo "<{}>"
echo "--- empty input (BSD does not run; GNU would run once without -r):"
printf '' | xargs echo "ran with no input"
echo "(finished, status $?)"
cd ..

--- splitting on blanks breaks names with spaces:
arg: with
arg: space.txt
--- -0 keeps each name whole:
arg: with space.txt
arg: plain.txt
--- find | xargs -0 is the safe pairing:
./plain.txt
./with space.txt
--- -n 2 groups two per call:
batch: 1 2
batch: 3 4
batch: 5
--- -I{} one run per input line:
<a>
<b>
--- empty input (BSD does not run; GNU would run once without -r):
(finished, status 0)


## 16. Command grouping: `{ ...; }` and `( ... )`

- `{ a; b; } > file` redirects the combined output of several commands in the current shell. Needs a space after `{` and a `;` (or newline) before `}`.
- `( a; b )` does the same in a subshell: `cd` and variable changes inside are discarded.
- Groups can be piped: `{ a; b; } | sort`.

In [19]:
%%bash
cd s01
{ echo "header"; echo "body"; echo "footer"; } > grouped.txt
cat grouped.txt
echo "--- braces keep changes, parentheses throw them away:"
v=before; { v=braces; }; echo "after { }: $v"
v=before; ( v=paren ); echo "after ( ): $v"
here=$(pwd | sed 's|.*/||'); ( cd / ); echo "cwd still in: $here == $(pwd | sed 's|.*/||')"
echo "--- one sort for several producers:"
{ echo c; echo a; echo b; } | sort | tr '\n' ' '; echo
cd ..

header
body
footer
--- braces keep changes, parentheses throw them away:
after { }: braces
after ( ): before
cwd still in: s01 == s01
--- one sort for several producers:
a b c 


## 17. `noclobber`: refuse to overwrite

- `set -o noclobber` (or `set -C`) makes `>` fail if the file already exists. `>>` still appends.
- `>|` forces an overwrite even under noclobber.
- Good safety net in scripts that write to user-given names. Shown in a fresh shell because it is a persistent option.

In [20]:
%%bash fresh
cd "$HOME"
echo one > keep.txt
set -o noclobber
echo two > keep.txt;  echo "status: $?"
echo "appending is still allowed:"; echo three >> keep.txt
echo forced >| keep.txt; echo "forced overwrite status: $?"
cat keep.txt
set +o noclobber

script: line 4: keep.txt: cannot overwrite existing file
status: 1
appending is still allowed:
forced overwrite status: 0
forced


## 18. Reading and writing the same file (the classic mistake)

- The shell opens (and empties) the output file BEFORE the command runs. `sort f > f` leaves `f` empty.
- Fix 1: write to a temp file then `mv` it over the original (portable, and atomic when both files are on the same disk).
- Fix 2: `sort -o f f` (sort handles it itself), or `sed -i ''` on BSD / `sed -i` on GNU (see 02-Text-Processing-Tools).
- `mv` over the original keeps the new file's permissions, not the old file's, so copy them if that matters.

In [21]:
%%bash
cd s01
printf 'pear\napple\nfig\n' > fruit.txt
sort fruit.txt > fruit.txt
echo "after 'sort f > f': $(wc -c < fruit.txt | tr -d ' ') bytes   <- data lost"
printf 'pear\napple\nfig\n' > fruit.txt
sort fruit.txt > fruit.tmp && mv fruit.tmp fruit.txt
echo "temp-file way:"; cat fruit.txt
printf 'pear\napple\nfig\n' > fruit.txt
sort -o fruit.txt fruit.txt
echo "sort -o way:"; cat fruit.txt
cd ..

after 'sort f > f': 0 bytes   <- data lost
temp-file way:
apple
fig
pear
sort -o way:
apple
fig
pear


## 19. Bad vs good: common redirection mistakes

- Quote variables in redirection targets: `> "$file"`. In bash an unquoted name with a space gives "ambiguous redirect". The message still shows below because `2>/dev/null` comes AFTER the failing `> $file` (redirections run left to right).
- `cmd 2>&1 > /dev/null` leaves errors visible, which is sometimes wanted (see section 03).
- `echo "text" > file | other` is pointless: stdout already went to the file, the pipe gets nothing.
- Prefer `printf '%s\n' "$x"` over `echo "$x"` when `$x` could start with `-` (bash `echo` takes `-n` as an option). Some `echo` versions, such as macOS `/bin/sh`, also treat backslashes in `$x` specially.

In [22]:
%%bash
cd s01
file="my report.txt"
echo "data" > $file 2>/dev/null || echo "unquoted target failed (ambiguous redirect)"
echo "data" > "$file" && echo "quoted target works: $(cat "$file")"
echo "--- pipe after a redirect receives nothing:"
echo "hello" > pipe_lost.txt | wc -c
echo "--- echo vs printf with a leading dash:"
x="-n"
echo "$x"; echo "(echo swallowed it as an option)"
printf '%s\n' "$x"
cd ..

script: line 3: $file: ambiguous redirect
unquoted target failed (ambiguous redirect)
quoted target works: data
--- pipe after a redirect receives nothing:
       0
--- echo vs printf with a leading dash:
(echo swallowed it as an option)
-n


## 20. Mini report from a pipeline

- Put the pieces together: heredoc as input data, a pipeline to summarise it, `tee` to save, a group to format it.
- Data: lines of `user,bytes`. Goal: total bytes per user, biggest first.
- `sort | uniq -c` and `awk` are covered in depth in 02-Text-Processing-Tools.

In [23]:
%%bash
cd s01
cat > usage.csv <<'EOF'
alice,120
bob,300
alice,80
carol,50
bob,100
EOF
{
  echo "user  total"
  echo "----  -----"
  awk -F, '{ sum[$1] += $2 } END { for (u in sum) print u, sum[u] }' usage.csv | sort -k2,2nr -k1,1
} | tee report.out
echo "--- saved copy has $(wc -l < report.out | tr -d ' ') lines"
cd ..

user  total
----  -----
bob 400
alice 200
carol 50
--- saved copy has 5 lines


## 21. Quick reference

| Want | Write |
|---|---|
| stdout to file (replace / append) | `cmd > f` / `cmd >> f` |
| stderr to file | `cmd 2> f` |
| both to one file | `cmd > f 2>&1` or `cmd &> f` (bash) |
| append both (3.2) | `cmd >> f 2>&1` |
| discard | `cmd > /dev/null 2>&1` |
| swap streams | `cmd 3>&1 1>&2 2>&3 3>&-` |
| pipe stderr too | `cmd 2>&1 \| next` |
| per-stage status | `${PIPESTATUS[@]}` (copy it at once) |
| text to stdin | `<<EOF`, `<<'EOF'`, `<<-EOF`, `<<< "text"` |
| command output as a file | `<( cmd )` |

Next: `02-Text-Processing-Tools` for what to put on the other side of the pipe, and `03-Files-And-System/01-File-System-Ops` for the files themselves.

In [24]:
%%bash
echo "--- swap stdout and stderr (3>&1 1>&2 2>&3 3>&-):"
swap() { { echo "I was stdout"; echo "I was stderr" >&2; } 3>&1 1>&2 2>&3 3>&-; }
swap 2>/dev/null          # the old stdout line is now on stderr, so it is hidden
echo "--- only the old stderr line survives above"

--- swap stdout and stderr (3>&1 1>&2 2>&3 3>&-):
I was stderr
--- only the old stderr line survives above
